    # Goals
    • Which input features are likely useful? 
        ○ past load values 
        ○ hour of day 
        ○ day of week 
        ○ weekend 
        ○ holiday indicator  -> external dataset with public holidays necessary
        ○ Month
        ○ season (Winter = Dec, Jan, Feb / Spring = Mar, Apr, May / Summer = Jun, Jul, Aug / Autumn = Sep, Oct, Nov)
        ○ weather variables (if available) -> maybe in an extension
        ○ renewable generation (solar / wind)
    • Do we need feature engineering such as lag features, rolling averages, or cyclical encodings? 
        ○ Lag features (t-1 = load 1 hour ago / t-2 = load 2 hours ago / t-24 = load 24 hours ago / t-168 = load 168 hours ago (= 1 week ago)
        ○ Rolling statistics (rolling mean 24h)
        - Cyclical encoding for time-based variables (hour of the day, day of week, month)

In [31]:
#!pip install holidays
import pandas as pd
import numpy as np
import holidays
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

In [32]:
df = pd.read_csv("../../data/austria/raw/austria_clean_final.csv")
df.head()

,timestamp,load_actual,load_forecast,solar,wind
0,2014-12-31 23:00:00+00:00,5946.0,6701.0,0.0,69.0
1,2015-01-01 00:00:00+00:00,5946.0,6701.0,0.0,69.0
2,2015-01-01 01:00:00+00:00,5726.0,6593.0,0.0,64.0
3,2015-01-01 02:00:00+00:00,5347.0,6482.0,0.0,65.0
4,2015-01-01 03:00:00+00:00,5249.0,6454.0,0.0,64.0


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50401 entries, 0 to 50400
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      50401 non-null  object 
 1   load_actual    50401 non-null  float64
 2   load_forecast  50401 non-null  float64
 3   solar          50401 non-null  float64
 4   wind           50401 non-null  float64
dtypes: float64(4), object(1)
memory usage: 1.9+ MB


In [34]:
# 2. Convert timestamp from string (object) to datetime objects
# utc=True handles the timezone offsets correctly during conversion
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# 3. Convert to local Austrian time
# This is crucial for correctly identifying local hours, weekends, and holidays
df['timestamp'] = df['timestamp'].dt.tz_convert('Europe/Vienna')

# 4. Set timestamp as the index
# A datetime index is required for shifting (lags) and resampling
df = df.set_index('timestamp').sort_index()

In [35]:
df.head()

,load_actual,load_forecast,solar,wind
timestamp,,,,
2015-01-01 00:00:00+01:00,5946.0,6701.0,0.0,69.0
2015-01-01 01:00:00+01:00,5946.0,6701.0,0.0,69.0
2015-01-01 02:00:00+01:00,5726.0,6593.0,0.0,64.0
2015-01-01 03:00:00+01:00,5347.0,6482.0,0.0,65.0
2015-01-01 04:00:00+01:00,5249.0,6454.0,0.0,64.0


In [36]:
# Define a function to create features for supervised learning

# 1. Function to map months to seasons
def get_season(month):
    """Maps month integers to season names."""
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

# 2. Main Feature Engineering Pipeline
def build_modeling_features(df):
    """
    Transforms clean data into a supervised learning dataset.
    Adds time features, cyclical encoding, holidays, lags, and the target.
    """
    
    # Create a copy to avoid SettingWithCopy warnings
    df = df.copy()
    
    # --- Time-Based Features ---
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    
    # Cyclical Encoding (Captures the circular nature of time)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Night / Day Indicator
    # Definition: Nacht = 22:00–05:59
    df['is_night'] = df['hour'].apply(lambda x: 1 if (x >= 22 or x < 6) else 0)
    
    # Weekend and Season
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['season'] = df['month'].apply(get_season)
    
    # Convert season to dummy variables (One-Hot Encoding)
    df = pd.get_dummies(df, columns=['season'], prefix='season')
    
    # --- Holiday Indicator (Austria) ---
    at_holidays = holidays.Austria()
    df['is_holiday'] = df.index.map(lambda x: 1 if x in at_holidays else 0).astype(int)
    
# --- NEW: Lag Features for all variables (Solar, Wind, Load) ---
    # Knowing yesterday's weather/load is key for predicting tomorrow
    for col in ['load_actual', 'solar', 'wind']:
        df[f'{col}_lag_24h'] = df[col].shift(24)
        df[f'{col}_lag_48h'] = df[col].shift(48)
        df[f'{col}_lag_168h'] = df[col].shift(168) # Same time last week
    
    # Short-term lags for load
    df['load_lag_1h'] = df['load_actual'].shift(1)
    df['load_lag_2h'] = df['load_actual'].shift(2)
    
    # --- Rolling Statistics ---
    # Mean load of the last 24 hours (shifted by 1 to avoid data leakage)
    df['load_rolling_mean_24h'] = df['load_actual'].shift(1).rolling(window=24).mean()
    
    # --- Target Definition ---
    # We want to forecast the load 24 hours ahead
    forecast_horizon = 24
    df['target_load_24h'] = df['load_actual'].shift(-forecast_horizon)
    
    # --- Data Cleaning ---
    # Convert any remaining booleans (from get_dummies) to integers (0/1)
    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)
            
    # Drop rows with NaNs (caused by lags at the start and the target shift at the end)
    df = df.dropna()

    # --- FINAL SELECTION: Drop redundant features and keep only relevant ones ---
    # We define exactly what we want to keep based on our correlation analysis
    relevant_features = [
        'load_actual', 'load_actual_lag_24h', 'load_actual_lag_48h', 'load_actual_lag_168h', 'load_lag_1h',
        'hour_cos', 'is_night', 'is_weekend', 'is_holiday',
        'solar', 'wind', 'season_Winter', 'season_Summer', 
        'target_load_24h'
    ]
    
    df = df[relevant_features]
    
    return df


def time_split(df, train_ratio=0.7, val_ratio=0.15):
    n = len(df)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]

    return train, val, test

In [43]:
# Apply the pipeline
df_model = build_modeling_features(df)

print("Supervised learning dataset created!")
print(f"Number of features: {len(df_model.columns)}")
print(f"Shape of the dataset: {df_model.shape}")

# --- 2. Final Export to CSV ---
output_path = "../../data/austria/raw/austria_modeling_master.csv"

df_model.to_csv(output_path, index=True)

print(f"Successfully exported master table to: {output_path}")

Supervised learning dataset created!
Number of features: 14
Shape of the dataset: (50209, 14)
Successfully exported master table to: ../../data/austria/raw/austria_modeling_master.csv


In [38]:
print(df_model.head(15))

                           load_actual  load_actual_lag_24h  \
timestamp                                                     
2015-01-08 00:00:00+01:00       6651.0               5973.0   
2015-01-08 01:00:00+01:00       6357.0               5568.0   
2015-01-08 02:00:00+01:00       6360.0               5642.0   
2015-01-08 03:00:00+01:00       6417.0               5547.0   
2015-01-08 04:00:00+01:00       6302.0               5846.0   
2015-01-08 05:00:00+01:00       6601.0               6134.0   
2015-01-08 06:00:00+01:00       7193.0               6658.0   
2015-01-08 07:00:00+01:00       7686.0               7557.0   
2015-01-08 08:00:00+01:00       8061.0               7301.0   
2015-01-08 09:00:00+01:00       8289.0               7589.0   
2015-01-08 10:00:00+01:00       7662.0               8031.0   
2015-01-08 11:00:00+01:00       7548.0               8168.0   
2015-01-08 12:00:00+01:00       7753.0               8316.0   
2015-01-08 13:00:00+01:00       7959.0               83

In [ ]:
train_df, val_df, test_df = time_split(df_model)
#defining features and target for modeling
target = "target_load_24h"

features = [col for col in train_df.columns if col != target]

def scale_data(train, val, test, feature_cols, target_col):
    train_c, val_c, test_c = train.copy(), val.copy(), test.copy()
    
    # Scaler für Features
    scaler_x = StandardScaler()
    # Separater Scaler für das Target (macht das "Un-Scaling" später einfacher)
    scaler_y = StandardScaler()

    # Features skalieren
    train_c[feature_cols] = scaler_x.fit_transform(train_c[feature_cols])
    val_c[feature_cols] = scaler_x.transform(val_c[feature_cols])
    test_c[feature_cols] = scaler_x.transform(test_c[feature_cols])

    # Target skalieren
    train_c[[target_col]] = scaler_y.fit_transform(train_c[[target_col]])
    val_c[[target_col]] = scaler_y.transform(val_c[[target_col]])
    test_c[[target_col]] = scaler_y.transform(test_c[[target_col]])

    return train_c, val_c, test_c, scaler_x, scaler_y

In [40]:
#windowing function to create sequences for LSTM
def create_sequences(df, feature_cols, target_col, input_window=168):
    X, y = [], []

    data = df[feature_cols].values
    target = df[target_col].values

    for i in range(len(df) - input_window):
        X.append(data[i:i+input_window])
        y.append(target[i+input_window])

    return np.array(X), np.array(y)

In [42]:
train_df, val_df, test_df, scaler_x, scaler_y = scale_data(
    train_df, val_df, test_df, features, target
)

X_train, y_train = create_sequences(train_df, features, target)
X_val, y_val = create_sequences(val_df, features, target)
X_test, y_test = create_sequences(test_df, features, target)

In [ ]:
# --- EXPORT FINAL DATASET ---

# Define the output path
# Using a relative path to the 'processed' data folder
output_path = "../../data/austria/raw/austria_modeling_table_24h_horizon.csv"

# Export to CSV
# We keep the index because it's our timestamp!
train_df.to_csv("../../data/austria/raw/train.csv", index=True)
val_df.to_csv("../../data/austria/raw/val.csv", index=True)
test_df.to_csv("../../data/austria/raw/test.csv", index=True)